# Hinglish Sentiment Classifier — MuRIL Fine-Tuning
**Upload this notebook to Kaggle and enable GPU (P100) before running.**

Also upload your `data/` folder as a Kaggle Dataset before running.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME   = "google/muril-base-cased"   # MuRIL: trained on 17 Indian languages + Hinglish
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 4
LR           = 2e-5
OUTPUT_DIR   = "./muril-hinglish-sentiment"

# Update this path to where you uploaded the data on Kaggle
DATA_DIR     = "/kaggle/input/hinglish-sentiment-data"   # <-- change if needed

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
train_df = pd.read_csv(f"{DATA_DIR}/sentiment_train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/sentiment_val.csv")

# Sanity check
print(f"Train: {len(train_df)} rows")
print(f"Val:   {len(val_df)} rows")
print(f"\nLabel distribution (train):")
print(train_df['sentiment'].value_counts())

# Drop nulls just in case
train_df = train_df.dropna(subset=["text", "label"])
val_df   = val_df.dropna(subset=["text", "label"])

In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────────────
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df[["text", "label"]])
val_ds   = Dataset.from_pandas(val_df[["text", "label"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

# HuggingFace Trainer expects "labels" not "label"
train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done.")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1_weighted": f1}

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,                   # warmup for first 10% of steps
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=50,
    fp16=True,                          # use GPU half-precision (faster)
    report_to="none"                    # disable wandb
)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training...")
trainer.train()

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\nFinal evaluation on validation set:")
results = trainer.evaluate()
print(results)

# Detailed classification report
predictions = trainer.predict(val_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = val_ds["labels"].numpy()

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["negative", "neutral", "positive"]))

In [ ]:
# ── Save Model ────────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved to {OUTPUT_DIR}")
print("Download this folder — you'll need it for the FastAPI deployment step.")

In [ ]:
# ── Quick Inference Test ──────────────────────────────────────────────────────
# Test the model with a few Hinglish examples before downloading
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    "yaar ye product ekdum bekar hai, waste of money",
    "bhai mast tha, bilkul maza aaya",
    "delivery thodi late thi but product theek hai",
    "kya bakwaas service hai, never ordering again",
    "bahut acha experience tha overall"
]

print("\nSample predictions:")
print("-" * 50)
for text in test_texts:
    result = classifier(text)[0]
    print(f"Text:  {text}")
    print(f"Pred:  {result['label']} (confidence: {result['score']:.2%})")
    print()